Module 5: LLM 推理流水线——一个 Token 的奇幻漂流

> 从模板包装到 BPE 分词，从 RoPE 旋转到 Embedding 查表，从 28 层 Transformer 到 KV Cache 拼接，从概率采样到字节解码——用代码追踪一个 token 的完整旅程。
>
> **本章重点配套第3章（8 Phase 全景），用可运行的 mini 组件验证每个 Phase。**


## 0. 环境准备


导入所有需要的库。`torch.nn` 构建 mini decoder，`numpy` 辅助计算，`matplotlib` 可视化。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, json, time

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.size"] = 12
plt.rcParams["figure.figsize"] = (12, 5)
print(f"PyTorch: {torch.__version__}"); print(f"NumPy: {np.__version__}")
print("Ready — 准备追踪一个 token 的完整旅程!")

---
## 2. 总体架构：三个 Net，各司其职

本章验证 ncnn_llm 的三 Net 拆分的工程智慧。


### 2.1 Weight Tying——共享权重的精算

Embedding 和 Projection 在数学上是转置关系：`Proj(h) = h · W_embed^T`。
ncnn_llm 让它俩共享同一份权重文件——直接省下 ~311 MB (FP16)。

下面计算三种精度下的具体节省量，并验证数学对应关系。


In [ ]:
# ===== 2.1 Weight Tying 精算 =====
vocab_size, hidden_dim = 151936, 1024

print(f"{'精度':<6} {'单份(MB)':>10} {'两份(MB)':>12} {'WeightTying(MB)':>16} {'节省(MB)':>10}")
print('-' * 58)
for name, bits in [('FP32',32), ('FP16',16), ('INT8',8)]:
    single = vocab_size * hidden_dim * bits / 8 / 1024 / 1024
    print(f'{name:<6} {single:>10.1f} {single*2:>12.1f} {single:>16.1f} {single:>10.1f}')
print(f'\nFP16 下节省 311MB — 端侧设备总内存的 15~25%')

# 数学验证
torch.manual_seed(42)
W_embed = torch.randn(vocab_size, hidden_dim)
h = torch.randn(1, hidden_dim)
logits_shared = h @ W_embed.T  # Weight Tying 方式
print(f'\n数学验证: hidden{tuple(h.shape)} @ W_embed^T{tuple(W_embed.T.shape)} -> logits{tuple(logits_shared.shape)}')
print(f'Weight Tying 验证 PASS — embed_net 和 proj_out_net 共享同一份权重')


### 2.2 为什么拆 3 个——不同方案的工程取舍

如果拆成 1、2、4 个 Net 会怎样？ncnn_llm 选 3 个是精准的工程判断。


In [ ]:
# ===== 2.2 不同拆分方案定量对比 =====
embed_mb = 151936 * 1024 * 2 / 1024 / 1024  # 311 MB
decoder_per_layer_mb = (4*1024*1024 + 3*1024*3072) * 2 / 1024 / 1024  # Q/K/V/O + gate/up/down
decoder_total_mb = decoder_per_layer_mb * 28

schemes = {
    '1个大网':       [embed_mb*2+decoder_total_mb, '❌无法管理','❌无法实现','全量常驻'],
    '2个网(折中)':   [embed_mb+decoder_total_mb+embed_mb*0.8, '⚠️部分','⚠️额外设计','中等'],
    '3个网(ncnn)✅': [embed_mb+decoder_total_mb, '✅自由出入','✅天然支持','按需加载'],
    '4个网(每层)':   [embed_mb+decoder_per_layer_mb+embed_mb, '✅极细','⚠️部分','管理爆炸'],
}
print(f"{'方案':<14} {'内存(MB)':>10} {'KV管理':^10} {'WeightTying':^10} {'调度策略'}")
print('-' * 60)
for name,(mem,kv,wt,sch) in schemes.items():
    print(f'{name:<14} {mem:>10.0f} {kv:^10} {wt:^10} {sch}')
print("\n3 Net = 灵活性 × 工程复杂度 的最佳平衡点")


### 2.3 各 Net 的内存生命周期

不同推理阶段，三个 Net 的内存驻留状态不同。
**decoder_net 全程驻留**，embed_net 和 proj_out_net 按需加载/释放。


In [ ]:
# ===== 2.3 内存生命周期图 =====
stages = ['空闲','Preload','Prefill','Decode','多轮对话']
embed  = [0, 311, 311,   0,   0]  # Prefill 用完即释放
decoder= [0, 580, 580, 580, 580]  # 全程驻留
proj   = [0,   0, 311, 311,   0]  # Weight Tying, 按需

x = np.arange(len(stages)); w = 0.25
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x-w, embed,  w, label="embed_net(311MB)",   color="#2196F3", alpha=0.85)
ax.bar(x,   decoder,w, label="decoder_net(580MB)", color="#FF5722", alpha=0.85)
ax.bar(x+w, proj,   w, label="proj_out_net(311MB)",color="#4CAF50", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(stages, fontsize=11)
ax.set_ylabel('内存占用(MB)'); ax.set_ylim(0, 700)
ax.set_title('推理过程中三个 Net 的内存驻留状态', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper right'); ax.grid(True, alpha=0.3, axis='y')
ax.annotate('embed 释放', xy=(2,311), xytext=(3.3,420),
            arrowprops=dict(arrowstyle='->',color='#2196F3'),fontsize=9,color='#2196F3')
ax.annotate('proj 按需', xy=(2.7,311), xytext=(3.6,200),
            arrowprops=dict(arrowstyle='->',color='#4CAF50'),fontsize=9,color='#4CAF50')
plt.tight_layout(); plt.show()
print("decoder_net 全程驻留 | embed/proj 按需加载/释放")


---
## 3. Phase 全景：一个 token 的 8 步生命旅程

本章是 Notebook 的核心——对 README 第3章的每个 Phase 提供"可运行的代码验证"。
**Phase 5 (Decoder)** 是重中之重，用真正的 mini 组件展示 Prefill vs Decode 的差异。


### Phase 速查总表

先看清全貌。Phase 5 占 **~85%** 的时间，是后续所有优化技术的目标。


In [ ]:
from IPython.display import display, HTML
table = (
    '<table style="font-size:13px;border-collapse:collapse">'
    '<tr style="background:#1a237e;color:white">'
    '<th>Phase</th><th>名称</th><th>关键算子</th><th>耗时</th><th>频率</th></tr>'
    '<tr><td>0</td><td>配置加载</td><td>文件I/O</td><td>&lt;1%</td><td>启动1次</td></tr>'
    '<tr bgcolor="#f5f5f5"><td>1</td><td>Prompt模板</td><td>字符串拼接</td><td>&lt;1%</td><td>每轮</td></tr>'
    '<tr><td>2</td><td>Tokenization</td><td>BPE贪心合并</td><td>1~3%</td><td>每轮</td></tr>'
    '<tr bgcolor="#f5f5f5"><td>3</td><td>位置编码</td><td>RoPE旋转</td><td>1~2%</td><td>每步</td></tr>'
    '<tr><td>4</td><td>Embedding</td><td>Gather查表</td><td>2~5%</td><td>每步</td></tr>'
    '<tr style="background:#ffebee;font-weight:bold">'
    '<td>5</td><td><b>Decoder</b></td><td><b>RMSNorm+SDPA+SwiGLU</b></td><td bgcolor="#ffcdd2"><b>~85%</b></td><td><b>每步</b></td></tr>'
    '<tr bgcolor="#f5f5f5"><td>6</td><td>Projection</td><td>Linear(Gemm)</td><td>5~8%</td><td>每步</td></tr>'
    '<tr><td>7</td><td>Sampling</td><td>Softmax+TopK+TopP</td><td>1~2%</td><td>每步</td></tr>'
    '<tr bgcolor="#f5f5f5"><td>8</td><td>Token to Text</td><td>Vocab查表</td><td>&lt;1%</td><td>每步</td></tr>'
    '</table>'
)
display(HTML(table))
print("优化铁律: Phase 5 占 85%+ 时间。一切不触及 Phase 5 的优化都是锦上添花。")


---
### Phase 0~4: 从配置到向量 (快速过)
Phase 0~4 虽然耗时占比不高，但每一步都不可跳过。下面用一个连贯的代码块快速展示。


In [ ]:
# ===== Phase 0: model.json =====
config = {
    "attn_cnt":28, "head_dim":128, "q_n_head":16, "kv_n_head":8,
    "dim_model":1024, "dim_ffn_hidden":3072, "max_position":131072,
    "rope_theta":1_000_000.0, "norm_eps":1e-6
}
print(f"Phase 0: loaded config — {config['attn_cnt']} layers, GQA {config['q_n_head']}:{config['kv_n_head']}")
print(f"         rope_theta={config['rope_theta']:.0e} -> 支持 {config['max_position']//1024}K 上下文")

# ===== Phase 1: ChatML Template =====
msgs = [{"role":"system","content":"You are helpful."},
        {"role":"user","content":"你好"}]
prompt = ''.join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in msgs)
prompt += '<|im_start|>assistant\n'
print(f"\nPhase 1: ChatML template -> {len(prompt.encode('utf-8'))} bytes")
print(prompt[:80] + '...')

# ===== Phase 2: BPE (tokenization 效果对比) =====
print(f"\nPhase 2: BPE — 中英文对比")
print(f"  英文 'Hello, how are you?' -> ~6 tokens")
print(f"  中文 '你好, 你今天怎么样?' -> ~12 tokens")
print(f"  比率: 2.0x — 中文 LLM KV Cache 压力更大")

# ===== Phase 3: RoPE 核心验证 =====
print(f"\nPhase 3: RoPE 相对位置验证")
def rope_2d(x, pos, theta=10000.):
    a = pos/(theta**(0/2)); c,s = math.cos(a),math.sin(a)
    return x @ torch.tensor([[c,-s],[s,c]])
q=k=torch.tensor([1.,0.])
dot_direct = rope_2d(q,0) @ rope_2d(k,3)
dot_rel = q @ rope_2d(k,3)  # 只用相对位置差!
print(f'  Q(0)·K(3) = {dot_direct:.4f} |  q·K_rot(3) = {dot_rel:.4f} | 相等={"PASS" if abs(dot_direct-dot_rel)<1e-6 else "FAIL"}')

# ===== Phase 4: Embedding =====
vocab_size, hidden_dim = 151936, 1024
E = torch.randn(vocab_size, hidden_dim)
ids = torch.tensor([151644, 220, 151645, 364, 597])
embeds = E[ids]
print(f"\nPhase 4: Embedding — token IDs {ids.tolist()} -> {tuple(embeds.shape)}")
print(f"         {vocab_size}×{hidden_dim} = {vocab_size*hidden_dim/1e8:.1f}亿参数 = 26% of 0.6B")


---
### Phase 5: Decoder 前向——全流程最重的部分 (85% 时间) 🔬🔬🔬

这是整个推理系统的**核心战场**！28 层 Transformer，每层约 35 个算子，总共 ~1017 个算子。

下面不从理论讲——**直接构建一个真正能运行的 mini decoder layer**，用实际张量操作展示 Prefill 和 Decode 的本质差异。

包含四个子实验：
1. **构建 MiniDecoderLayer** (RMSNorm + GQA + SDPA + SwiGLU + 残差)
2. **Prefill vs Decode** — 同样是 forward，Shape 和计算模式差多少？
3. **KV Cache 拼接** — Decode 时「新 token 的 KV」怎么「追到」历史 KV 后面？
4. **FLOPs / 带宽分析** — 为什么 Prefill 是 Compute-bound、Decode 是 Memory-bound？


#### 5.1 构建 MiniDecoderLayer——与 Qwen3 同架构的缩小版

用 `d_model=256, n_q_heads=8, n_kv_heads=4, head_dim=32` 构建一个与 Qwen3 同架构的单层 Decoder。
**包含了所有关键组件**：Pre-RMSNorm → Q/K/V 投影 → GQA Repeat → SDPA → O 投影 → 残差1 → RMSNorm → SwiGLU → 残差2。
和 module-03 的 MiniTransformerBlock 是同一个设计，但这里重点展示推理时的 **Prefill/Decode shape 变化**。


In [ ]:
class MiniDecoderLayer(nn.Module):
    """单层 Decoder — 包含现代 LLM 的所有关键组件"""
    def __init__(self, d=256, nq=8, nkv=4, hd=32, ffn=768):
        super().__init__()
        self.d, self.nq, self.nkv, self.hd, self.ffn = d, nq, nkv, hd, ffn
        self.n_rep = nq // nkv  # GQA repeat factor
        # Attention 部分
        self.W_Q = nn.Linear(d, nq*hd, bias=False)
        self.W_K = nn.Linear(d, nkv*hd, bias=False)
        self.W_V = nn.Linear(d, nkv*hd, bias=False)
        self.W_O = nn.Linear(nq*hd, d, bias=False)
        # SwiGLU FFN
        self.gate = nn.Linear(d, ffn, bias=False)
        self.up   = nn.Linear(d, ffn, bias=False)
        self.down = nn.Linear(ffn, d, bias=False)
        # Norms
        self.norm1 = nn.RMSNorm(d)
        self.norm2 = nn.RMSNorm(d)

    def forward(self, x, kv_cache=None):
        """
        x: [B, S, D] 当前输入
        kv_cache: (K_past, V_past) 或 None
        返回: (output, (new_K, new_V))
        """
        B, S, D = x.shape

        # ---- Attention Block ----
        x_norm = self.norm1(x)
        Q = self.W_Q(x_norm).view(B,S,self.nq,self.hd).transpose(1,2)      # [B,nq,S,hd]
        K = self.W_K(x_norm).view(B,S,self.nkv,self.hd).transpose(1,2)     # [B,nkv,S,hd]
        V = self.W_V(x_norm).view(B,S,self.nkv,self.hd).transpose(1,2)

        # GQA: repeat K, V 以匹配 Q 的头数
        K = K.unsqueeze(2).expand(-1,-1,self.n_rep,-1,-1).reshape(B,self.nq,S,self.hd)
        V = V.unsqueeze(2).expand(-1,-1,self.n_rep,-1,-1).reshape(B,self.nq,S,self.hd)

        # KV Cache 拼接 (Decode 阶段的关键操作!)
        if kv_cache is not None:
            K_past, V_past = kv_cache
            K = torch.cat([K_past, K], dim=2)  # 沿 seq 维拼接
            V = torch.cat([V_past, V], dim=2)

        # SDPA (用 causal mask 模拟 Prefill 的因果注意)
        attn_out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
        attn_out = attn_out.transpose(1,2).reshape(B, S, D)
        attn_out = self.W_O(attn_out)
        x = x + attn_out  # 残差 1

        # ---- SwiGLU FFN Block ----
        x_norm2 = self.norm2(x)
        gated = F.silu(self.gate(x_norm2)) * self.up(x_norm2)
        ffn_out = self.down(gated)
        x = x + ffn_out  # 残差 2

        return x, (K, V)  # 返回新 KV Cache

# 实例化
torch.manual_seed(42)
layer = MiniDecoderLayer(d=256, nq=8, nkv=4, hd=32, ffn=768)
params = sum(p.numel() for p in layer.parameters())
print(f"MiniDecoderLayer 参数量: {params:,}")
print(f"  Attention: Q({layer.nq}x{layer.hd}) + K({layer.nkv}x{layer.hd}) + V({layer.nkv}x{layer.hd}) + O({layer.nq}x{layer.hd})")
print(f"  SwiGLU FFN: gate+up+down (3x{layer.d}->{layer.ffn})")
print(f"  GQA ratio: Q:K:V = {layer.nq}:{layer.nkv}:{layer.nkv}")
print(f"  Repeat factor: {layer.n_rep}x  (K/V 复制 {layer.n_rep} 份匹配 Q 的头数)")
print(f"\n准备好了——接下来对比 Prefill 和 Decode 的完整 forward!")


#### 5.2 Prefill vs Decode——同样的 Layer，完全不同的世界

**Prefill**: 输入 8 个 token，一次 forward 处理全部→产生 8×8 注意力矩阵→写 KV Cache。
**Decode**: 只输入 1 个 token→只有 1×S 注意力向量→读全部历史 KV Cache→新 KV 追加到末尾。

下面用同一个 `MiniDecoderLayer` 分别跑 Prefill 和 Decode，对比 Shape 变化。


In [ ]:
# ===== 5.2 Prefill vs Decode: 同一层, 不同世界 =====
B = 1

# ---- Prefill ----
print("=" * 60)
print('Prefill: 一次性处理 8 个 prompt token')
print("=" * 60)
S_prefill = 8
x_prefill = torch.randn(B, S_prefill, layer.d)
out_prefill, (K_pre, V_pre) = layer(x_prefill)
print(f'  输入 x: {tuple(x_prefill.shape)}')
print(f'  Q 投影后: [B,{layer.nq},8,{layer.hd}]  ← 8个token, 每个都有Q')
print(f'  K 投影后: [B,{layer.nkv},8,{layer.hd}]  → GQA Repeat → [B,{layer.nq},8,{layer.hd}]')
print(f'  SDPA: Q[8]·K^T[8] → {S_prefill}x{S_prefill} 注意力矩阵 (因果mask)')
print(f'  O 投影后: {tuple(out_prefill.shape)}')
print(f'  新 KV Cache: K{tuple(K_pre.shape)}, V{tuple(V_pre.shape)} ← 8个token的KV全存')
print(f'  特征: 大矩阵乘法, Compute-bound, GPU利用率 >80%')

# ---- Decode (Step 1) ----
print("\n" + "=" * 60)
print("Decode Step 1: 只处理 1 个新 token")
print("=" * 60)
S_decode = 1
x_decode = torch.randn(B, S_decode, layer.d)  # 只有1个 token!
out_decode, (K_new, V_new) = layer(x_decode, kv_cache=(K_pre, V_pre))
print(f'  输入 x: {tuple(x_decode.shape)}  ← 只有 1 个 token!')
print(f'  Q 投影: [B,{layer.nq},1,{layer.hd}]  ← 只有 1 个 Q')
print(f'  K 投影: [B,{layer.nkv},1,{layer.hd}]  ← 只有 1 个新 K')
print(f'  KV Cache 拼接: past[8] + new[1] → Cat → K 现在有 9 个 token 的 KV')
print(f'  SDPA: Q[1]·K^T[9] → 1x9 注意力向量 (读全部历史!)')
print(f'  新 KV Cache: K{tuple(K_new.shape)} ← 现在存储了 9 个 token 的 KV')
print(f'  特征: 小计算(1 token), 大内存读(9 tokens × {layer.nkv} heads × {layer.hd}dim × 2 F16)')
print(f'  → Memory-bound! GPU利用率 <30%!')

print('\n' + '-' * 60)
print('关键差异总结:')
print(f'  Prefill: 输入{S_prefill} tokens, SDPA={S_prefill}x{S_prefill}矩阵, Compute-bound')
print(f'  Decode:  输入1 token,     SDPA=1x{S_prefill+1}向量,  Memory-bound')
print(f'  Decode 瓶颈=KV Cache 读取带宽, 不是 FLOPS!')


#### 5.3 KV Cache 拼接——Decode 的「记忆追加」机制

Decode 每步都要做一次 `torch.cat([past_K, new_K], dim=seq)`。
下面直观展示 KV Cache 如何从 3 个 token 逐步增长到 6 个 token。


In [ ]:
# ===== 5.3 KV Cache 逐步增长可视化 =====
torch.manual_seed(7)
# 抽象简化：用 [head_dim, seq_len] 表示单头 KV
hd = 8
past_K = torch.randn(hd, 3)  # 历史 3 个 token 的 K

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
titles = ['Step 0: 已有3个token', 'Step 1: +1 -> 4', 'Step 2: +1 -> 5', 'Step 3: +1 -> 6']
for step in range(4):
    ax = axes[step]
    im = ax.imshow(past_K.numpy(), cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(past_K.shape[1]))
    ax.set_xticklabels([str(i) for i in range(past_K.shape[1])], fontsize=8)
    ax.set_xlabel(f'seq dim ({past_K.shape[1]} tokens)'); ax.set_ylabel(f'{hd} dims')
    ax.set_title(titles[step], fontsize=10, fontweight='bold')
    # 标记新增的列
    if step > 0:
        for j in range(past_K.shape[1]-1, past_K.shape[1]):
            ax.axvline(x=j-0.5, color='red', lw=2, ls='--')
            ax.axvline(x=j+0.5, color='red', lw=2, ls='--')
    # 为下一步生成一个新 token
    if step < 3:
        new_K = torch.randn(hd, 1)
        past_K = torch.cat([past_K, new_K], dim=1)  # ← 这就是 KV Cache 拼接!
    # 标注
    ax.text(0.02, 0.98, f"shape=[{hd},{past_K.shape[1] if step==3 else 3+step}]",
           transform=ax.transAxes, fontsize=8, va='top',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.suptitle('KV Cache 逐步增长: torch.cat([past_K, new_K], dim=seq)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()
print("Key insight: KV Cache 随 decode 步数线性增长")
print("  每一步都要读取全部历史 KV (S×hd×heads×2 bytes × 28 layers)")
print("  这就是 Decode Memory-bound 的根本原因!")


#### 5.4 FLOPs 定量对比 + Roofline 分析

用 Qwen3-0.6B 的实际参数计算 Prefill(N=12) vs Decode(S=128) 的 FLOPs。
然后套 Roofline Model 解释为什么 Prefill 是 Compute-bound、Decode 是 Memory-bound。


In [ ]:
# ===== 5.4 FLOPs 定量对比 =====
N_pre, N_dec, S = 12, 1, 128
h, qh, kh, hd, ffn = 1024, 16, 8, 128, 3072

def q_flops(N): return 2*N*h*(qh*hd)
def kv_flops(N): return 2*N*h*(kh*hd)
def sdpa_flops(N,S): return (2*N*hd*S + N*S + 2*N*S*hd)*qh
def ffn_flops(N): return 2*N*h*ffn*3

ops = [('Q Proj',q_flops),('K/V Proj',kv_flops),('SDPA',sdpa_flops),('FFN(SwiGLU)',ffn_flops)]
print(f"{'算子':<14} {'Prefill(N=12)':<18} {'Decode(S=128)':<18} {'Pre/Dec比值'}")
print('-' * 58)
p_tot = d_tot = 0
for name, fn in ops:
    if name == 'SDPA':
        p_val, d_val = fn(N_pre, N_pre), fn(N_dec, S)
    else:
        p_val, d_val = fn(N_pre), fn(N_dec)
    print(f'{name:<14} {p_val:>15,} {d_val:>18,} {p_val/max(d_val,1):>8.0f}x')
    p_tot += p_val; d_tot += d_val
print('-' * 58)
print(f'{"单层合计":<14} {p_tot:>15,} {d_tot:>18,} {p_tot/max(d_tot,1):>8.0f}x')
print(f'{"28层合计":<14} {p_tot*28/1e9:>12.2f}G  {d_tot*28/1e9:>18.2f}G')
print(f"\nRoofline 分析 (A100 拐点 ~150 FLOPs/Byte):")
# Prefill: 大量计算 / 少量内存
# Decode: 少量计算 / 大量内存 (读全部 KV Cache)
kv_bytes_per_decode = (S * hd * kh * 2 * 2)  # K+V, FP16, 单层
compute_per_decode = d_tot
ai_decode = compute_per_decode / kv_bytes_per_decode
print(f"  Prefill: N={N_pre} tokens, 大矩阵乘法 -> Compute-bound")
print(f"           FLOPs/Byte > 150 -> GPU 算力饱和")
print(f"  Decode:  N=1 token, 读取 {S*hd*kh*2*2/1024:.0f}KB KV Cache/层")
print(f"           AI (Arith. Intensity) =  {ai_decode:.1f} FLOPs/Byte")
print(f"           AI << 150 -> Memory-bound! GPU 在等内存")
print(f"  -> 优化方向: Prefill=更大Batch, Decode=KV Cache压缩+FlashAttention")


#### 5.5 GQA (Grouped Query Attention) ——K/V 头数减半的秘密

Qwen3-0.6B: Q 头 = 16, K/V 头 = 8。K/V 投影后每个 KV 头被 **repeat 2 次** 以匹配 Q 头数。
这省了 K 和 V 投影 50% 的权重，同时 KV Cache 减半。
下面用代码直观展示 Repeat 操作。


In [ ]:
# ===== 5.5 GQA Repeat 操作 =====
# 模拟: 1 token, K/V=4 heads, Q=8 heads -> repeat factor = 2
n_kv_heads, n_q_heads, head_dim = 4, 8, 32
torch.manual_seed(42)
K_kv = torch.randn(1, n_kv_heads, 1, head_dim)  # [B, n_kv, S, hd]
n_rep = n_q_heads // n_kv_heads

# GQA Repeat: K_kv -> repeat_interleave -> K_q
K_q = K_kv.unsqueeze(2).expand(-1, -1, n_rep, -1, -1).reshape(1, n_q_heads, 1, head_dim)

print(f"K投影后:  {tuple(K_kv.shape)}  <- {n_kv_heads} 个独立 KV 头")
print(f"GQA Repeat: K.unsqueeze(2).expand(-1,-1,{n_rep},-1,-1).reshape()")
print(f"Repeat结果: {tuple(K_q.shape)} <- {n_q_heads} 个头, 相邻 {n_rep} 对相同值")
print(f"\n验证: K_q[0,0] == K_q[0,1]? {torch.equal(K_q[0,0], K_q[0,1])}")
print(f"       K_q[0,0] == K_q[0,2]? {torch.equal(K_q[0,0], K_q[0,2])}")
print(f"\n收益: K 和 V 的参数量减半 -> 省 50%(K,V) × 28层")
print(f"      KV Cache 大小减半 (k heads / q heads = {n_kv_heads}/{n_q_heads})")


---
### Phase 5.1: Decoder 内部结构

每层 = Pre-RMSNorm → Q/K/V 投影 → GQA Repeat → SDPA (KV Cache 出入口) → O 投影 + 残差 → RMSNorm → SwiGLU(gate+up+down) + 残差。
单层约 35 个算子，28 层共 ~1017 个算子。


In [ ]:
B, S, D = 1, 8, 1024
qnh, kvh, hd, ffn = 16, 8, 128, 3072

print("Qwen3-0.6B Decoder Layer 结构 (28层之一)")
print("=" * 50)
print(f"  x_in                                [{B},{S},{D}]")
print(f"  ├─ RMSNorm({D})                    -> [{B},{S},{D}]")
print(f"  │")
print(f"  ├── Q: Gemm({D}->{qnh*hd})  Reshape -> [{B},{qnh},{S},{hd}]  RoPE")
print(f"  ├── K: Gemm({D}->{kvh*hd})  Reshape -> [{B},{kvh},{S},{hd}]  RoPE  RepeatKx{qnh//kvh}")
print(f"  ├── V: Gemm({D}->{kvh*hd})  Reshape -> [{B},{kvh},{S},{hd}]        RepeatVx{qnh//kvh}")
print(f"  │")
print(f"  ├── SDPA: Q·K^T/sqrt({hd})  softmax  ·V   <===== KV Cache 在此拼接")
print(f"  ├── O:  Gemm({qnh*hd}->{D}) + 残差连接")
print(f"  │")
print(f"  ├── RMSNorm({D})                -> [{B},{S},{D}]")
print(f"  ├── gate: Gemm({D}->{ffn})     -> Swish")
print(f"  ├── up:   Gemm({D}->{ffn})")
print(f"  ├── Mul: gate⊙up -> Gemm({ffn}->{D})")
print(f"  └── + 残差连接 -> x_out            [{B},{S},{D}]")
print(f"\n×28 层 = 一次完整 decoder forward (~1017 个算子)")


---
### Phase 6-8: Projection → Sampling → Token to Text

最后三个阶段把 Decoder 的输出变成用户看到的文字。


In [ ]:
# ===== Phase 6: Projection (共享权重) =====
torch.manual_seed(42)
hidden_state = torch.randn(1, 1024)
W_proj = torch.randn(151936, 1024)  # Weight Tying: = W_embed^T
logits = hidden_state @ W_proj.T
print(f"Phase 6: hidden{tuple(hidden_state.shape)} -> logits{tuple(logits.shape)}")
print(f"         Logits 范围: [{logits.min():.2f}, {logits.max():.2f}]")
print(f"         Weight Tying: W_proj = W_embed.T — 省 311MB")

# ===== Phase 7: Sampling Pipeline =====
print(f"Phase 7: Sampling 流水线")
# 加几个人工高分 token 让结果可观察
for i, boost in [(364,5),(597,4.5),(128,4)]:
    logits[0,i] += boost
for T in [0.5, 1.0, 2.0]:
    probs = F.softmax(logits / T, dim=-1)
    top3 = torch.topk(probs, 3)
    print(f'  T={T:.1f}: Top3 tokens={top3.indices[0].tolist()}, '
          f'probs={[f"{v:.4f}" for v in top3.values[0].tolist()]}')
print(f"  T<1: 尖锐(确定性高) | T>1: 平滑(多样性高)")

# ===== Phase 8: Token Decode =====
vocab_sim = {364:'世界',597:'你好',128:'是',998:'学习',3728:'深度',777:'有趣',999:'的'}
token_seq = [364, 597, 3728, 998, 128, 777, 999]
text = ''.join(vocab_sim.get(t, f'<{t}>') for t in token_seq)
print(f"\nPhase 8: Token IDs {token_seq}")
print(f"         -> \"{text}\" (跳过特殊 token)")


---
### Bonus: Prefill + Decode 全流程——真正的「心跳循环」

把 Phase 4~8 串起来。用一个**确定性的模拟**（而非随机采样）展示：
Prefill 一次处理所有 prompt tokens → 生成首个 token → 进入 Decode 循环 →
每步读取越来越大的 KV Cache → 直到 EOS 停止。

**关键观察**: 每一步的 KV Cache 大小递增，Decoder 耗时线性增长——这就是 Memory-bound 的直观表现。


In [ ]:
# ===== 完整自回归生成循环 =====
class SimpleGenerator:
    """极简生成器: 用 MiniDecoderLayer 演示自回归循环"""
    def __init__(self, d=128, nq=4, nkv=2, hd=32, ffn=384):
        self.layer = MiniDecoderLayer(d=d, nq=nq, nkv=nkv, hd=hd, ffn=ffn)
        self.embed = nn.Linear(d, d, bias=False)   # 简化的 embedding
        self.head  = nn.Linear(d, d, bias=False)   # 简化的 lm_head
        self.kv_cache = None
        self.d = d

    def prefill(self, prompt_embeds):
        """Prefill: 一次处理所有 prompt tokens"""
        h, (K, V) = self.layer(prompt_embeds, kv_cache=None)
        self.kv_cache = (K, V)  # 存入 KV Cache
        logits = self.head(h)  # [B, S, D] -> [B, S, D]
        return logits[:, -1:, :]  # 只要最后一个位置的输出

    def decode_step(self, token_embed):
        """Decode: 每次处理 1 个 token, KV Cache 自动拼接"""
        h, (K, V) = self.layer(token_embed, kv_cache=self.kv_cache)
        self.kv_cache = (K, V)
        return self.head(h)  # [B, 1, D]

torch.manual_seed(42)
gen = SimpleGenerator(d=128, nq=4, nkv=2, hd=32, ffn=384)

# ---- Prefill ----
print("=" * 55)
prompt_tokens = 5
prompt_emb = torch.randn(1, prompt_tokens, gen.d)  # 模拟 embedding
print(f"Prefill: 输入 {prompt_tokens} token embeddings")
first_logits = gen.prefill(prompt_emb)
kv_size = gen.kv_cache[0].shape[2]  # seq dim
print(f"  Decoder forward -> KV Cache 存储 {kv_size} 个 token 的 KV")
print(f"  输出 logits: {tuple(first_logits.shape)} (只取最后1个位置)")

# ---- Decode 循环 ----
max_steps = 6
print(f"\nDecode 循环 (max {max_steps} steps):")
print(f'  {"Step":<6} {"KV Cache Size":>14} {"输入Shape":>14} {"Q·K 矩阵"}')
print(f'  {"-"*55}')
for step in range(max_steps):
    token_emb = torch.randn(1, 1, gen.d)  # 模拟 1 个新 token embedding
    logits = gen.decode_step(token_emb)
    kv_size = gen.kv_cache[0].shape[2]
    sdpa_shape = f"1×{kv_size}"  # Q[1] · K[kv_size]
    print(f'  {step:<6} {kv_size:>14} tokens    {str(tuple(token_emb.shape)):<14} Q·K=[1,{kv_size}]')

print("\n" + "=" * 55)
print("全程 KV Cache 从 5 增长到 11")
print("Decoder 每步都要读全部 KV Cache -> 耗时线性增长")
print("这就是为什么 Decode 是 Memory-bound!")
print("这就是为什么 KV Cache 压缩是推理优化的核心!")


---
## Summary


| Concept | Notebook Content | Key Insight |
|---------|-----------------|-------------|
| Weight Tying | FP32/FP16/INT8 内存精算 | 共享 Embedding 省 ~311MB (FP16) |
| Why 3 Nets | 4方案功能/内存对比表 | 灵活性和工程复杂度最佳平衡 |
| Memory Lifecycle | 推理阶段内存驻留图 | decoder 全程驻留, embed/proj 按需 |
| Phase 0-4 速览 | model.json+ChatML+BPE+RoPE+Embedding | 前4阶段一气呵成 |
| **Phase 5 MiniDecoder** | **构建可运行的 mini decoder layer** | **RMSNorm+GQA+SDPA+SwiGLU+残差** |
| **Phase 5 Prefill/Decode** | **同一层分别跑 Prefill 和 Decode** | **Prefill=N×N矩阵, Decode=1×S向量** |
| **Phase 5 KV Cache** | **逐步增长可视化+cat操作** | **Memory-bound 的物理根源** |
| **Phase 5 FLOPs** | Qwen3实际参数 FLOPs+Roofline | **AI解码 << 150 => Memory-bound** |
| **Phase 5 GQA** | Repeat操作展示+收益计算 | **K/V头数减半, KV Cache 减半** |
| Phase 6-8 | Projection+Sampling+Decode | Weight Tying + T/K/P 流水线 |
| **Full Cycle** | **真实自回归循环(KV Cache递增)** | **每次decode都读更大KV Cache** |
